# RAG critic and report search

The team's `FieldVision/rag-pipeline` in the agents:
- **critic**: code checks the county's silage share against Quick Stats history and decides pass/warn/fail and the next action; Nemotron Ultra only explains, citing NASS report passages;
- **report search**: the field Q&A can quote NASS Crop Progress reports and harvest-date handbooks, with source and page.

Cells 1-4 are free (local embeddings). Cells 5-7 call Nemotron (a few cents).

In [ ]:
# 1. Setup
import os, sys, json
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline tests (fake critic and index, no credits)
from fieldvision_agents.fieldvision_bridge.tests.run import run_all
run_all()

In [ ]:
# 3. The report index: one search, Maryland plus national passages
from fieldvision_agents.fieldvision_bridge import make_report_search
from fieldvision_agents.fieldvision_bridge.config import RAG_PIPELINE
print(json.loads((RAG_PIPELINE / 'rag_index' / 'meta.json').read_text()))
search = make_report_search()
for h in search('When is corn for silage usually harvested in Maryland?', '24033', k=3):
    print(f"{h['source']} p.{h['page']} ({h['state']}, {h['year']}): {h['text'][:160]!r}")

In [ ]:
# 4. The critic's code checks alone on Beltsville 2022 (no LLM)
from fieldvision_agents.fieldvision_bridge import FieldVisionToolbox, make_rag_critic
tb = FieldVisionToolbox(critic=make_rag_critic(use_llm=False))
tb.classify('24033', 2022, 'standard')
tb.aggregate('24033', 2022)
verdict = tb.run_critic('24033', 2022)
print(verdict['overall'], '->', verdict['suggested_action'])
for c in verdict['checks']:
    print(f"  {c['check']:26s} {c['status']:5s} {c.get('why', '')}")

In [ ]:
# 5. The Nemotron supervisor with the real critic (Lightning runs, Ultra explains the verdict)
from fieldvision_agents.react.supervisor import NemotronClient, run_county
fv = FieldVisionToolbox(critic=make_rag_critic(), reports=search)
result = run_county('24033', 2022, fv, NemotronClient())
print(json.dumps({k: result[k] for k in ['run_id', 'status', 'policies', 'critic', 'summary', 'tokens']}, indent=1))
critic = fv.run_critic('24033', 2022)
print('\ncritic explained by:', critic.get('explained_by'), critic.get('explain_error', ''))
print(critic['summary'])
for f in critic['flags']:
    print(' -', f['check'], f['status'], '|', f['why'], '|', [(e.get('source'), e.get('page')) for e in f.get('evidence', [])])

In [ ]:
# 6. Field questions that need NASS reports as context
from fieldvision_agents.fieldvision_bridge import ask_fields
for q in ['How did the 2022 Maryland corn silage harvest progress compare with normal?',
          'Was BARC-252 harvested at a usual time for corn silage in Maryland?']:
    a = ask_fields(fv, '24033', 2022, result, q)
    print('Q:', q, '\nA:', a['answer'], '\n   tools=', a['tools_used'], 'grounded=', a['grounded'], '\n')

In [ ]:
# 7. The report, now with the real critic's verdict in the evidence
from IPython.display import Markdown, display
from fieldvision_agents.reporter.reporter import write_report
rep = write_report(result)
display(Markdown(rep['report_md'])); print(rep['grounding'])